In [ ]:
"""
PSIms Data Preparation & Configuration Module
Automates Excel to CSV conversion and manages flexible file paths
Handles Windows administrative access and user-specified directories

Author: Healthcare Analytics Team
Version: 2.0 - Automated Excel Processing
"""

import pandas as pd
import os
import sys
import json
from pathlib import Path
from datetime import datetime
import openpyxl
import shutil

# =====================================================
# CONFIGURATION MANAGEMENT
# =====================================================

class PSImsConfig:
    """Manages all file paths and configuration for PSIms"""
    
    def __init__(self, config_file='psims_config.json'):
        self.config_file = config_file
        self.config = self.load_config()
    
    def load_config(self):
        """Load configuration from JSON file or create default"""
        if os.path.exists(self.config_file):
            print(f"📁 Loading configuration from {self.config_file}")
            with open(self.config_file, 'r') as f:
                return json.load(f)
        else:
            print(f"⚠️  No config file found. Creating default configuration...")
            return self.create_default_config()
    
    def create_default_config(self):
        """Create default configuration"""
        default_config = {
            "paths": {
                "excel_input_folder": r"C:\PSIms\Input",
                "csv_data_folder": r"C:\PSIms\Data",
                "output_folder": r"C:\PSIms\Output"
            },
            "excel_files": {
                "main_data_file": "Batch_1_5000_HCPs_Profiles_22July2025.xlsx",
                "sheet_mapping": {
                    "Jan25": "Jan25.csv",
                    "Feb25": "Feb25.csv",
                    "March25": "March25.csv",
                    "April25": "April25.csv",
                    "May25": "May25.csv",
                    "June25": "June25.csv",
                    "July25": "July25.csv",
                    "PI": "pi.csv",
                    "Clinics": "clinics.csv",
                    "Publications": "Publications.csv",
                    "Trials": "Trials.csv",
                    "Academic_Association": "Academic_association.csv",
                    "Digital": "digital.csv",
                    "Healthcare_Platforms": "Healthcare.csv",
                    "Awards": "awards.csv",
                    "Press": "press.csv",
                    "Events": "events.csv",
                    "Associations": "association.csv"
                }
            },
            "processing": {
                "auto_backup": True,
                "timestamp_outputs": True,
                "validate_data": True
            }
        }
        
        self.save_config(default_config)
        return default_config
    
    def save_config(self, config=None):
        """Save configuration to JSON file"""
        if config is None:
            config = self.config
        
        with open(self.config_file, 'w') as f:
            json.dump(config, f, indent=4)
        print(f"✓ Configuration saved to {self.config_file}")
    
    def update_path(self, path_type, new_path):
        """Update a specific path in configuration"""
        if path_type in self.config['paths']:
            self.config['paths'][path_type] = new_path
            self.save_config()
            print(f"✓ Updated {path_type} to: {new_path}")
        else:
            print(f"❌ Unknown path type: {path_type}")
    
    def get_path(self, path_type):
        """Get a specific path from configuration"""
        return self.config['paths'].get(path_type, '')
    
    def create_directories(self):
        """Create all required directories if they don't exist"""
        print("\n📁 Checking/Creating Required Directories...")
        
        for path_name, path_value in self.config['paths'].items():
            try:
                Path(path_value).mkdir(parents=True, exist_ok=True)
                print(f"   ✓ {path_name}: {path_value}")
            except PermissionError:
                print(f"   ❌ Permission denied for {path_name}: {path_value}")
                print(f"      Please run as Administrator or choose a different path")
            except Exception as e:
                print(f"   ❌ Error creating {path_name}: {e}")
    
    def print_config(self):
        """Display current configuration"""
        print("\n" + "="*70)
        print("CURRENT CONFIGURATION")
        print("="*70)
        print("\n📂 File Paths:")
        for key, value in self.config['paths'].items():
            print(f"   {key:.<30} {value}")
        
        print(f"\n📄 Excel File: {self.config['excel_files']['main_data_file']}")
        print(f"   Sheets to process: {len(self.config['excel_files']['sheet_mapping'])}")
        print("="*70)


# =====================================================
# EXCEL TO CSV CONVERTER
# =====================================================

class ExcelToCSVConverter:
    """Handles conversion of Excel sheets to CSV files"""
    
    def __init__(self, config):
        self.config = config
        self.conversion_log = []
    
    def find_excel_file(self):
        """Find the Excel file in the input folder"""
        input_folder = self.config.get_path('excel_input_folder')
        excel_file_name = self.config.config['excel_files']['main_data_file']
        excel_path = os.path.join(input_folder, excel_file_name)
        
        if os.path.exists(excel_path):
            print(f"✓ Found Excel file: {excel_path}")
            return excel_path
        else:
            # Try to find any Excel file (.xlsx, .xls, .xlsm) in the folder
            print(f"⚠️  Specified file not found: {excel_path}")
            print(f"   Searching for Excel files in {input_folder}...")
            
            # Check if folder exists
            if not os.path.exists(input_folder):
                print(f"❌ Folder does not exist: {input_folder}")
                return None
            
            # List all files in the folder
            all_files = os.listdir(input_folder)
            print(f"   Files in folder: {all_files}")
            
            # Try different Excel extensions
            excel_extensions = ['*.xlsx', '*.xls', '*.xlsm', '*.XLSX', '*.XLS', '*.XLSM']
            excel_files = []
            
            for ext in excel_extensions:
                found = list(Path(input_folder).glob(ext))
                excel_files.extend(found)
            
            if excel_files:
                excel_path = str(excel_files[0])
                print(f"✓ Found Excel file: {excel_path}")
                return excel_path
            else:
                print(f"❌ No Excel files (.xlsx, .xls, .xlsm) found in {input_folder}")
                print(f"   Please ensure your Excel file is in this folder")
                return None
    
    def list_excel_sheets(self, excel_path):
        """List all sheets in the Excel file"""
        try:
            wb = openpyxl.load_workbook(excel_path, read_only=True)
            sheets = wb.sheetnames
            wb.close()
            print(f"\n📊 Found {len(sheets)} sheets in Excel file:")
            for i, sheet in enumerate(sheets, 1):
                print(f"   {i}. {sheet}")
            return sheets
        except Exception as e:
            print(f"❌ Error reading Excel file: {e}")
            return []
    
    def convert_sheet_to_csv(self, excel_path, sheet_name, output_csv_path):
        """Convert a single Excel sheet to CSV"""
        try:
            # Read the sheet
            df = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')
            
            # Save as CSV
            df.to_csv(output_csv_path, index=False, encoding='utf-8')
            
            log_entry = {
                'sheet': sheet_name,
                'csv_file': os.path.basename(output_csv_path),
                'rows': len(df),
                'columns': len(df.columns),
                'status': 'success',
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            
            self.conversion_log.append(log_entry)
            print(f"   ✓ {sheet_name} → {os.path.basename(output_csv_path)} ({len(df)} rows, {len(df.columns)} cols)")
            
            return True
        
        except Exception as e:
            log_entry = {
                'sheet': sheet_name,
                'csv_file': output_csv_path,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            self.conversion_log.append(log_entry)
            print(f"   ❌ Failed to convert {sheet_name}: {e}")
            return False
    
    def convert_all_sheets(self, excel_path=None):
        """Convert all configured sheets to CSV"""
        if excel_path is None:
            excel_path = self.find_excel_file()
        
        if not excel_path:
            print("❌ Cannot proceed without Excel file")
            return False
        
        print("\n🔄 Starting Excel to CSV Conversion...")
        print("="*70)
        
        sheet_mapping = self.config.config['excel_files']['sheet_mapping']
        csv_folder = self.config.get_path('csv_data_folder')
        
        # List available sheets
        available_sheets = self.list_excel_sheets(excel_path)
        
        print(f"\n📝 Converting {len(sheet_mapping)} sheets...")
        
        success_count = 0
        for sheet_name, csv_filename in sheet_mapping.items():
            if sheet_name in available_sheets:
                output_path = os.path.join(csv_folder, csv_filename)
                if self.convert_sheet_to_csv(excel_path, sheet_name, output_path):
                    success_count += 1
            else:
                print(f"   ⚠️  Sheet '{sheet_name}' not found in Excel file")
        
        print("\n" + "="*70)
        print(f"✓ Conversion Complete: {success_count}/{len(sheet_mapping)} sheets successful")
        print("="*70)
        
        # Save conversion log
        self.save_conversion_log()
        
        return success_count > 0
    
    def save_conversion_log(self):
        """Save conversion log to file"""
        if not self.conversion_log:
            return
        
        log_df = pd.DataFrame(self.conversion_log)
        output_folder = self.config.get_path('output_folder')
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        log_file = os.path.join(output_folder, f'conversion_log_{timestamp}.csv')
        
        log_df.to_csv(log_file, index=False)
        print(f"\n📋 Conversion log saved: {log_file}")


# =====================================================
# FILE ACCESS CHECKER
# =====================================================

def check_windows_access():
    """Check if the script has necessary permissions"""
    print("\n🔐 Checking Windows Permissions...")
    
    # Check if running as admin
    try:
        is_admin = os.getuid() == 0
    except AttributeError:
        import ctypes
        try:
            is_admin = ctypes.windll.shell32.IsUserAnAdmin() != 0
        except:
            is_admin = False
    
    if is_admin:
        print("   ✓ Running with Administrator privileges")
    else:
        print("   ⚠️  Not running as Administrator")
        print("      Some paths may require elevated privileges")
    
    return is_admin


# =====================================================
# INTERACTIVE CONFIGURATION SETUP
# =====================================================

def interactive_setup():
    """Guide user through configuration setup"""
    print("\n" + "="*70)
    print("PSIms CONFIGURATION SETUP")
    print("="*70)
    
    print("\nWould you like to:")
    print("1. Use default paths")
    print("2. Customize paths")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    config = PSImsConfig()
    
    if choice == '2':
        print("\n📁 Let's set up your custom paths...")
        
        # Excel input folder
        excel_input = input(f"\nExcel input folder [{config.get_path('excel_input_folder')}]: ").strip()
        if excel_input:
            config.update_path('excel_input_folder', excel_input)
        
        # Excel filename
        current_filename = config.config['excel_files']['main_data_file']
        excel_filename = input(f"\nExcel filename [{current_filename}]: ").strip()
        if excel_filename:
            config.config['excel_files']['main_data_file'] = excel_filename
            config.save_config()
            print(f"✓ Updated Excel filename to: {excel_filename}")
        
        # CSV data folder
        csv_folder = input(f"\nCSV data folder (where model reads from) [{config.get_path('csv_data_folder')}]: ").strip()
        if csv_folder:
            config.update_path('csv_data_folder', csv_folder)
        
        # Output folder
        output_folder = input(f"\nOutput folder (for results) [{config.get_path('output_folder')}]: ").strip()
        if output_folder:
            config.update_path('output_folder', output_folder)
    
    # Create directories
    config.create_directories()
    
    # Display final configuration
    config.print_config()
    
    return config


# =====================================================
# MAIN EXECUTION
# =====================================================

def main():
    """Main execution function"""
    print("="*70)
    print("PSIms DATA PREPARATION TOOL")
    print("Excel to CSV Automation with Flexible Path Configuration")
    print("Version 2.0")
    print("="*70)
    
    # Check Windows permissions
    check_windows_access()
    
    # Check if config exists
    if not os.path.exists('psims_config.json'):
        print("\n⚠️  No configuration file found.")
        config = interactive_setup()
    else:
        print("\n✓ Configuration file found")
        choice = input("Use existing configuration? (y/n): ").strip().lower()
        
        if choice == 'n':
            config = interactive_setup()
        else:
            config = PSImsConfig()
            config.print_config()
    
    # Convert Excel to CSV
    print("\n" + "="*70)
    print("EXCEL TO CSV CONVERSION")
    print("="*70)
    
    converter = ExcelToCSVConverter(config)
    
    # Ask user if they want to proceed
    proceed = input("\nProceed with conversion? (y/n): ").strip().lower()
    
    if proceed == 'y':
        success = converter.convert_all_sheets()
        
        if success:
            print("\n✅ Data preparation complete!")
            print(f"   CSV files are ready in: {config.get_path('csv_data_folder')}")
            print(f"   You can now run the PSIms analysis pipeline.")
        else:
            print("\n❌ Conversion failed. Please check the errors above.")
    else:
        print("\n⚠️  Conversion cancelled by user")
    
    print("\n" + "="*70)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Process interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()